# Correlation-Informed Quantum Learning

In [ ]:
from pathlib import Path
from pprint import pprint

from corr_inf_quant_learning import SimulationConfig, run_benchmark
from corr_inf_quant_learning.plotting import plot_benchmark

#state and target property
N_QUBITS = 100
K_ROTATIONS = 50
SUBSYSTEM_SIZE = 7       #used only for PURITY
PROPERTY = "MAGIC"       #"PURITY" or "MAGIC"
METHODS = ("AGHDO-CS","AGHDO-BI")
#METHODS = ("CS","BI")

#measurement budget and averaging
PILOT_SHOTS = 0
POST_SHOTS = 10000
REPETITIONS = 20
EVALUATION_POINTS = 9
SEED = 7
COUNT_PILOT_COST = True

#bound-informed acquisition and scalable estimators
BI_EXPLORATION = 0.001       #eta: uniform share of q_t
BI_COMMUTING_BIAS = 1     #PURITY softmax: >1 favors commuting
BI_SOFTMAX_TEMPERATURE = 0.01  # PURITY and MAGIC softmax: tau (->0 argmax, ->inf uniform over candidates)
BI_PAULI_FLOOR = 0.0        #PURITY softmax: coverage weight floor for every P_A
BI_PURITY_WEIGHT = 1.0      #PURITY: lambda_purity
BI_GENERATOR_WEIGHT = 1   #PURITY: lambda_gen, online generator-learning reward
BI_UPDATE_EVERY = 1         #PURITY: recompute q_t every this many shots
BI_CLIP_BOUND = "POINT"
BI_CANDIDATES = 100         #MAGIC: candidates scored per shot
AGHDO_FIT_STEPS = 12
QNS_PROPERTY_SAMPLES = 4096

#progress bar
PROGRESS_DETAILS = False

# Figure controls
ERROR_MODE = "relative"   #"absolute" or "relative"
LOG_X = True
LOG_Y = True
SMOOTH_WINDOW = 1
SHOW_ERROR_BAND = True


In [ ]:
config = SimulationConfig(
    n_qubits=N_QUBITS,
    k_rotations=K_ROTATIONS,
    subsystem_size=SUBSYSTEM_SIZE,
    property=PROPERTY,
    methods=METHODS,
    pilot_shots=PILOT_SHOTS,
    post_shots=POST_SHOTS,
    repetitions=REPETITIONS,
    evaluation_points=EVALUATION_POINTS,
    seed=SEED,
    count_pilot_cost=COUNT_PILOT_COST,
    bi_exploration=BI_EXPLORATION,
    bi_commuting_bias=BI_COMMUTING_BIAS,
    bi_candidates=BI_CANDIDATES,
    bi_purity_weight=BI_PURITY_WEIGHT,
    bi_generator_weight=BI_GENERATOR_WEIGHT,
    bi_softmax_temperature=BI_SOFTMAX_TEMPERATURE,
    bi_pauli_floor=BI_PAULI_FLOOR,
    bi_update_every=BI_UPDATE_EVERY,
    bi_clip_bound=BI_CLIP_BOUND,
    aghdo_fit_steps=AGHDO_FIT_STEPS,
    qns_property_samples=QNS_PROPERTY_SAMPLES,
    progress_details=PROGRESS_DETAILS,
)

result = run_benchmark(config)
print(f"Exact {PROPERTY.lower()}: {result.truth:.8g}")
print(f"Subsystem: {result.subsystem}")
print(f"Pilot cost: {result.pilot_cost} shots")
pprint(result.final_summary(ERROR_MODE))


In [ ]:
repository_root = Path.cwd()
if not (repository_root / "pyproject.toml").exists():
    repository_root = repository_root.parent
output_stem = repository_root / "figures" / f"cluster_{PROPERTY.lower()}_{ERROR_MODE}"
figure, axis, saved_files = plot_benchmark(
    result,
    error_mode=ERROR_MODE,
    log_x=LOG_X,
    log_y=LOG_Y,
    smooth_window=SMOOTH_WINDOW,
    show_band=SHOW_ERROR_BAND,
    output_stem=output_stem,
)
print("Saved:", *(str(path) for path in saved_files), sep="\n  " )
figure